# RecruiterFit AI Candidate Ranking

This notebook shows the full CPU-only ranking pipeline and also supports a fast rerun path from the saved top-1000 retrieval pool.

In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.preprocess
import src.io_utils
import src.retrieval
import src.scoring

importlib.reload(src.preprocess)
importlib.reload(src.io_utils)
importlib.reload(src.retrieval)
importlib.reload(src.scoring)

from src.preprocess import load_candidates, normalize_candidates, prefilter_raw_active_candidates, prefilter_raw_role_candidates
from src.io_utils import read_job_description
from src.retrieval import retrieve_top_k
from src.scoring import rank_candidates, save_submission


In [2]:
# Set this to True only when you want to rebuild from candidates.jsonl.
# Keep it False for quick reruns using the saved top-1000 retrieval file.
RUN_FULL_PIPELINE = True

candidate_path = PROJECT_ROOT / "data/raw/candidates.jsonl"
processed_path = PROJECT_ROOT / "data/processed/cleaned_candidates.csv"
job_path = PROJECT_ROOT / "data/raw/job_description.docx"
saved_retrieved_path = PROJECT_ROOT / "outputs/retrieved_top_1000_from_previous_run.csv"
retrieved_output_path = PROJECT_ROOT / "outputs/retrieved_top_1000.csv"
output_path = PROJECT_ROOT / "outputs/final_top_100_with_reasoning.csv"

top_k = 1000
raw_role_prefilter_k = 3500
tfidf_prefilter_k = None


## Full Pipeline Path

Runs preprocessing and FAISS retrieval from the full raw `candidates.jsonl`. This is slower, but it shows the complete system.

In [3]:
if RUN_FULL_PIPELINE:
    job_description = read_job_description(job_path)

    raw = load_candidates(candidate_path)
    print("raw", raw.shape)

    raw = prefilter_raw_active_candidates(raw)
    print("after active prefilter", raw.shape)

    raw = prefilter_raw_role_candidates(raw, job_description, top_n=raw_role_prefilter_k)
    print("after raw HashingVectorizer role prefilter", raw.shape)

    candidates = normalize_candidates(raw)
    processed_path.parent.mkdir(parents=True, exist_ok=True)
    candidates.to_csv(processed_path, index=False)

    retrieved = retrieve_top_k(
        candidates,
        job_description,
        top_k=top_k,
        cache_dir=PROJECT_ROOT / "data/processed",
        tfidf_prefilter_k=None,
    )

    retrieved_output_path.parent.mkdir(parents=True, exist_ok=True)
    retrieved.to_csv(retrieved_output_path, index=False)

    print("processed", candidates.shape)
    print("retrieved", retrieved.shape)


raw (100000, 8)
after active prefilter (93533, 8)
after raw HashingVectorizer role prefilter (3500, 9)
Creating candidate embeddings. This is slow only the first time for this processed file.


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Saved candidate embeddings cache: c:\Users\shrey\Documents\Codex\2026-06-14\recruiters-go-through-hundreds-of-profiles\data\processed\candidate_embeddings_1b7288df771333b7.npy


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

processed (3500, 5)
retrieved (1000, 7)


## Fast Saved-Retrieval Path

Uses the previously saved top-1000 pool, avoiding preprocessing and embedding reruns.

In [4]:
if not RUN_FULL_PIPELINE:
    retrieved = pd.read_csv(saved_retrieved_path)
    print("loaded retrieved", retrieved.shape)


## Final Ranking And Submission

Ranks the retrieved pool locally and saves exactly 100 rows in the required format.

In [5]:
job_description = read_job_description(job_path)
ranked = rank_candidates(retrieved, job_description)
submission = save_submission(ranked, output_path, top_n=100)

print(f"Saved top 100 submission to: {output_path}")
print("submission shape:", submission.shape)
submission.head(20)


Saved top 100 submission to: c:\Users\shrey\Documents\Codex\2026-06-14\recruiters-go-through-hundreds-of-profiles\outputs\final_top_100_with_reasoning.csv
submission shape: (100, 4)


,candidate_id,rank,score,reasoning
0,CAND_0007460,1,0.860555,AI Engineer with 4.7 years of experience and s...
1,CAND_0030550,2,0.857925,AI Research Engineer with 5.2 years of experie...
2,CAND_0002770,3,0.840356,AI Research Engineer with 5.7 years of experie...
3,CAND_0006833,4,0.804051,AI Research Engineer with 5.7 years of experie...
4,CAND_0070333,5,0.802584,AI Research Engineer with 4.7 years of experie...
5,CAND_0068351,6,0.802223,Lead AI Engineer with 6.4 years of experience ...
6,CAND_0011327,7,0.797018,AI Research Engineer with 6.3 years of experie...
7,CAND_0005538,8,0.796914,Senior AI Engineer with 5.9 years of experienc...
8,CAND_0048558,9,0.787751,Data Scientist with 6.7 years of experience an...
9,CAND_0043958,10,0.787127,AI Research Engineer with 6.7 years of experie...
